# 난수 생성



NumPy 2.x에서 권장하는 방식은 **`Generator` API** (`default_rng`)  
기존의 `np.random.seed()` + 전역 상태 방식은 **비권장(legacy)**

Generator 방식이란 무엇인가?
> 난수 상태를 객체(Generator)에 담아 쓰는 방식이다.  
> 즉, np.random.default_rng(sedd)로 개별 난수 생성기 인스턴스를 만들고, 그 객체에서만 난수를 뽑는다.  

  - 상태 위치
      - Generator: 상태가 rng 객체 안에 있음 (독립적)
      - Legacy: 상태가 np.random 전역에 있음 (공유됨)
  - 재현성 제어
      - Generator: 필요한 곳마다 rng를 넘기면 그 부분만 동일하게 재현
      - Legacy: 전역 상태라 다른 코드가 호출되면 결과가 쉽게 바뀜
  - 안전성/병렬성
      - Generator: 스레드/병렬 작업에서 충돌 없이 독립 사용 가능
      - Legacy: 전역 상태 공유로 충돌 위험




### 비교

| 구분 | 권장 (Generator) | 비권장 (Legacy) |
|------|-----------------|----------------|
| 생성 | `rng = np.random.default_rng(42)` | `np.random.seed(42)` |
| 실수 | `rng.random(5)` | `np.random.random(5)` |
| 정수 | `rng.integers(0, 10, size=5)` | `np.random.randint(0, 10, 5)` |
| 정규분포 | `rng.normal(0, 1, size=5)` | `np.random.normal(0, 1, 5)` |
| 장점 | 독립적 상태, 스레드 안전 | - |

> **seed**: 난수의 시작점. 같은 seed → 같은 난수 시퀀스 (재현 가능)


  np.random.randint(low, high=None, size=None, dtype=int)

  - 기능: 정수 난수를 균일분포에서 생성
  - 인자:
      - low: (필수) 최소값. 포함됨.
      - high: (선택) 최대값. 포함되지 않음. 생략 시 0~low 범위.
      - size: (선택) 출력 크기. 예: 5, (2,3)
      - dtype: (선택) 결과 타입. 기본 int
  - 예:
      - np.random.randint(0, 10, size=5) → 0~9 정수 5개

  np.random.normal(loc=0.0, scale=1.0, size=None)

  - 기능: 정규분포(가우시안) 난수를 생성
  - 인자:
      - loc: (선택) 평균(μ), 기본 0.0
      - scale: (선택) 표준편차(σ), 기본 1.0
      - size: (선택) 출력 크기
  - 예:
      - np.random.normal(0, 1, size=5) → 평균 0, 표준편차 1 난수 5개

In [1]:
import numpy as np

In [7]:
print(np.random.randint(1,10,(2,5))) # size에 이런식으로 shape 넣기도 가능!

# 최신 Generator 방식
rng = np.random.default_rng(seed = 42)
print(rng.integers(1,10,(2,5)))

[[2 2 3 1 6]
 [8 4 6 9 5]]
[[1 7 6 4 4]
 [8 1 7 2 1]]


In [ ]:
# 난수 생성 (최신 Generator API)


# seed=42로 재현 가능한 난수 생성기 만들고 난 후 

# 균등 분포 [0, 1) 실수 5개
print(f"uniform [0,1):  {rng.random(5).round(3)}")

# 정수 난수 [0, 10) 범위 5개
print(f"integers [0,10): {rng.integers(0, 10, size=5)}")
print(f"integers [0,10): {rng.integers(0, 10, size=(2,3))}")
# 정규분포 (평균=0, 표준편차=1) 5개
print(f"normal(0,1):    {rng.normal(0, 1, size=5).round(3)}")
print(f"normal(10,1000):    {rng.normal(10, 1000, size=5).round(3)}")

# 배열 셔플 (순서 무작위 섞기)
arr = np.arange(1, 6)
rng.shuffle(arr)
print(f"\nshuffle:        {arr}")

# 랜덤 선택 (복원/비복원 추출)
choices = rng.choice([10, 20, 30, 40, 50], size=3, replace=False)
print(f"choice(비복원): {choices}")

uniform [0,1):  [0.044 0.154 0.683 0.745 0.968]
integers [0,10): [4 4 3 9 3]
integers [0,10): [[0 4 7]
 [1 4 1]]
normal(0,1):    [-0.84  -0.824  0.651  0.743  0.543]
normal(10,1000):    [-655.51   242.161  126.686  228.689  881.429]

shuffle:        [1 4 5 3 2]
choice(비복원): [30 10 40]


In [18]:
rng.choice([1,10,100,10000,100000],size = (2,3),replace=True) #replace = True -> 복원추출

array([[     1,    100,  10000],
       [ 10000,    100, 100000]])

# 조건처리

조건 처리: where와 clip

배열의 값을 조건에 따라 변환하는 함수들

| 함수 | 설명 | 예시 |
|------|------|------|
| `np.where(조건, 참값, 거짓값)` | 조건에 따라 값 선택 | 음수를 0으로 대체 |
| `np.clip(arr, min, max)` | 범위 밖의 값을 경계값으로 제한 | 0~10 범위로 제한 |

```
여기서 참값 = 조건이 True인 위치에 들어갈 값  
거짓값 = 조건이 False인 위치에 들어갈 값

np.where(arr < 0, 0, arr):
  [-5, -1, 0, 3, 7] → [0, 0, 0, 3, 7]

np.clip(arr, 0, 10):
  [-5, -1, 0, 3, 7, 12] → [0, 0, 0, 3, 7, 10]
```

np.clip() 함수는 array의 범위를 강제로 제한하는 함수이다. 최소, 최대 바깥은 최소와 최댓값으로 바꾼다.  

In [23]:
arr = np.array([-5, -1, 0, 3, 7, 12])

print(np.where((arr>10)|(arr<0),100,arr))

# np.where: 레이블 생성에도 활용할 수 있다. 
labels = np.where(arr >= 0, '양수입니다', '음수입니다')
print(f"where(레이블): {labels}")

[100 100   0   3   7 100]
where(레이블): ['음수입니다' '음수입니다' '양수입니다' '양수입니다' '양수입니다' '양수입니다']


In [ ]:
arr1 = np.array([-5, -1, 0, 100, 7, 12])
np.where(arr1>50,np.nan,arr1)

# 이렇게도 가능함!

array([-5., -1.,  0., nan,  7., 12.])

# NaN 처리

NaN이 포함된 배열에 np.mean()과 같은 일반 함수를 쓰면 그 결과도 NaN으로 반환하기 때문에, NaN을 대체하거나, **전용함수**를 사용하면 된다

| 상황 | 일반 함수 | NaN 안전 함수 |
|------|----------|-------------|
| 평균 | `np.mean()` → NaN | `np.nanmean()` → NaN 제외 후 계산 |
| 합계 | `np.sum()` → NaN | `np.nansum()` |
| 최대 | `np.max()` → NaN | `np.nanmax()` |
| 표준편차 | `np.std()` → NaN | `np.nanstd()` |

In [ ]:
arr = np.array([1.0, np.nan, 3.5, np.nan, 5.0])

print(f"\nNaN 여부:  {np.isnan(arr)}")    # [False  True False  True False] # Nan인 부분에 Ture를 반환함
print(f"NaN 개수:  {np.isnan(arr).sum()}") # 2


# 일반 함수 vs NaN 안전 함수
print(f"\nmean()    = {np.mean(arr)}")     # nan — NaN이 전파됨!
print(f"nanmean() = {np.nanmean(arr)}")    # 3.1666... — NaN 제외 후 계산
print(f"nansum()  = {np.nansum(arr)}")     # 9.5
print(f"nanmax()  = {np.nanmax(arr)}")     # 5.0

# NaN 대체
filled_zero = np.nan_to_num(arr, nan=0.0)     # np.nan_to_num() 함수는 nan값을 유한한 숫자로 치환하는 함수이다. 즉, 결측값을 계산가능한 값으로 '대체' 할 때 쓰인다. 
filled_mean = np.where(np.isnan(arr), np.nanmean(arr), arr)  # NaN → 평균
print(f"\nNaN→0 대체:   {filled_zero}")
print(f"NaN→평균 대체: {filled_mean.round(2)}")


NaN 여부:  [False  True False  True False]
NaN 개수:  2

mean()    = nan
nanmean() = 3.1666666666666665
nansum()  = 9.5
nanmax()  = 5.0

NaN→0 대체:   [1.  0.  3.5 0.  5. ]
NaN→평균 대체: [1.   3.17 3.5  3.17 5.  ]


# array 결합

### 결합 함수

| 함수 | 설명 | 비유 |
|------|------|------|
| `np.concatenate([])` | 지정 축을 따라 결합 | 범용 |
| `np.vstack([])` | 세로(수직) 결합 | 행 추가 |
| `np.hstack([])` | 가로(수평) 결합 | 열 추가 |

(2차원이상에서는) np.concatenate(axis=0)과 np.vstack()이 같은 효과겠다! (보통 비슷한고 1차원 배열의 결합에서만 다르다. )

In [30]:
a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])

print("a:")
print(a)
print("\nb:")
print(b)

# vstack: 세로(위아래) 결합 — 행 추가
print(f"\nvstack (세로 결합):\n{np.vstack([a, b])}")
#   [[1, 2], [3, 4], [5, 6], [7, 8]]  shape (4, 2)

# hstack: 가로(좌우) 결합 — 열 추가
print(f"\nhstack (가로 결합):\n{np.hstack([a, b])}")
#   [[1, 2, 5, 6], [3, 4, 7, 8]]  shape (2, 4)

# concatenate: 축 지정 결합 (vstack/hstack의 범용 버전)
print(f"\nconcatenate(axis=0):\n{np.concatenate([a, b], axis=0)}")  # = vstack
print(f"\nconcatenate(axis=1):\n{np.concatenate([a, b], axis=1)}")  # = hstack

a:
[[1 2]
 [3 4]]

b:
[[5 6]
 [7 8]]

vstack (세로 결합):
[[1 2]
 [3 4]
 [5 6]
 [7 8]]

hstack (가로 결합):
[[1 2 5 6]
 [3 4 7 8]]

concatenate(axis=0):
[[1 2]
 [3 4]
 [5 6]
 [7 8]]

concatenate(axis=1):
[[1 2 5 6]
 [3 4 7 8]]


In [ ]:
np.vstack([a,b]) @ np.hstack([a,b]) # shape(4,4)

array([[  7,  10,  19,  22],
       [ 15,  22,  43,  50],
       [ 23,  34,  67,  78],
       [ 31,  46,  91, 106]])

# array 분할

### 분할 함수

| 함수 | 설명 |
|------|------|
| `np.split()` | 균등 분할 |
| `np.vsplit()` | 행 방향 분할 |
| `np.hsplit()` | 열 방향 분할 |

````

  np.split(a, n) 형태로 쓰면:

  - 배열을 n개의 동일한 크기 조각으로 나눕니다.

  vsplit / hsplit에서 “몇 개로 나눌지”를 숫자로 줄 때
  에러가 안 나려면 해당 축 길이가 그 숫자로 정확히 나누어떨어져야 합니다.

  - np.vsplit(a, n) → 행 개수가 n으로 나누어떨어져야 함
  - np.hsplit(a, n) → 열 개수가 n으로 나누어떨어져야 함

In [42]:
arr = np.arange(1, 13).reshape(3, 4)
print(f"원본:\n{arr}")

# n개로 균등하게 split
print(f'3개 split: {np.split(arr,3)}')

# hsplit: 열 방향으로 2등분
left, right = np.hsplit(arr, 2)
print(f"\nhsplit - 왼쪽:\n{left}")
print(f"hsplit - 오른쪽:\n{right}")

# vsplit: 행 방향으로 3등분
top, mid, bot = np.vsplit(arr, 3)
print(f"\nvsplit: {top.ravel()}, {mid.ravel()}, {bot.ravel()}")

원본:
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]
3개 split: [array([[1, 2, 3, 4]]), array([[5, 6, 7, 8]]), array([[ 9, 10, 11, 12]])]

hsplit - 왼쪽:
[[ 1  2]
 [ 5  6]
 [ 9 10]]
hsplit - 오른쪽:
[[ 3  4]
 [ 7  8]
 [11 12]]

vsplit: [1 2 3 4], [5 6 7 8], [ 9 10 11 12]


In [ ]:
np.hsplit(arr, 4)
# 이 경우 axis=1 즉, 축의 갯수가 n=4와 딱 맞아 떨어진다 (4/4) 그러므로 나누기가 가능함!
# 물론, 이 경우 객체 선언을 하기 위해서는 총 4개가 필요하겠지(예를 들어 a1 a2 a3 a4..)

[array([[1],
        [5],
        [9]]),
 array([[ 2],
        [ 6],
        [10]]),
 array([[ 3],
        [ 7],
        [11]]),
 array([[ 4],
        [ 8],
        [12]])]

# Numpy가 편리한점
### Numpy는 for 루프 없이도 사용 가능하다던데, 그게 무슨 의미인가?


In [ ]:
N =100
list_a = list(range(N))
list_b = list(range(N))

list_c = [x + y for x, y in zip(list_a, list_b)]  # for 루프 필요


arr_a = np.arange(N)
arr_b = np.arange(N)
arr_c = arr_a + arr_b   # 벡터 연산 (한 줄!) 그리고 C언어 기반이기 때문에 훨씬 빠르다1=!

# array 연습

In [51]:
scores = np.array([
    [88,76,91],
    [92,85,79],
    [70,90,86],
    [60,65,72],
    [99,94,96]
])

In [52]:
scores

array([[88, 76, 91],
       [92, 85, 79],
       [70, 90, 86],
       [60, 65, 72],
       [99, 94, 96]])

In [53]:
test = np.where(scores>90, '합격','불합격')
print(test)

[['불합격' '불합격' '합격']
 ['합격' '불합격' '불합격']
 ['불합격' '불합격' '불합격']
 ['불합격' '불합격' '불합격']
 ['합격' '합격' '합격']]


In [58]:
과목평균 = np.mean(scores, axis=0)
학생평균 = np.mean(scores, axis=1)

print(f'과목별 평균:{과목평균} / 학생별 평균:{학생평균}')

과목별 평균:[81.8 82.  84.8] / 학생별 평균:[85.         85.33333333 82.         65.66666667 96.33333333]


In [64]:
excellent_mask = 학생평균 >= 85
excellent_idx = np.where(excellent_mask)[0]
print(excellent_mask)
print(excellent_idx)

[ True  True False False  True]
[0 1 4]


In [66]:
scores[excellent_mask]

array([[88, 76, 91],
       [92, 85, 79],
       [99, 94, 96]])